In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import PanDBIO
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource']


# DB-Specific

In [3]:
from lib import setlistfm
mio   = setlistfm.MusicDBIO(verbose=False, mkDirs=False)
apiio = setlistfm.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant SetListFM DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/SetListFM


# Master Files

In [136]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localArtistsData   = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsData".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = Series(dtype='object') #mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [138]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Artist Search:       {0}".format(len(localArtists.get())))
print("   Local Artist Search Data:  {0}".format(len(localArtistsData.get())))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

SetListFM Search Results
   Local Artist Search:       28859
   Local Artist Search Data:  0
   Global Master Search:      34949
   Global Master Search Data: 0
   Search Artists:            34949
   Errors:                    263
   Known Summary IDs:         0


# Search For New Artists

In [ ]:
mio   = setlistfm.MusicDBIO(verbose=False,local=False,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [ ]:
from musicdb import PanDBIO
from gate import IOStore

pdbio = PanDBIO()
mmeDF = pdbio.getData().sort_values(by=["Counts", "Albums"], ascending=False)

ios     = IOStore()
mdbio   = ios.get(db="MusicBrainz")
refData = mdbio.data.getSummaryRefData().to_dict()

mbIDData = mmeDF[mmeDF["MusicBrainz"].notna()][["ArtistName", "MusicBrainz"]]
mbIDData["MBRef"] = mbIDData["MusicBrainz"].apply(refData.get).apply(lambda x: x.split('/')[-1] if isinstance(x,str) else None)

searchedForMasterArtists = masterArtists.get()
artistNamesToGet = mbIDData[~mbIDData["MusicBrainz"].isin(searchedForMasterArtists.keys())]

print("{0} Search Results".format(db))
print("   Known Master Artist Names:    {0}".format(mbIDData.shape[0]))
print("   Known Spotify Artist Names:   {0}".format(len(searchedForMasterArtists)))
print("   Artist Names To Get:          {0}".format(len(artistNamesToGet)))

del pdbio
del mmeDF
del refData
del mbIDData

#   Artist Names To Get:          793373

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

## Run @ 3-4 PM every day

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "11:50pm")
n    = 0
maxN = 1400

searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
nErr = []
for i,(idx,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["ArtistName"]
    artistID = row["MusicBrainz"]
    mbID = row["MBRef"]
    if searchedForMasterArtists.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoResults(artistName=artistName, mbID=mbID)
    if response is None:
        print("Error ==> {0}".format((artistID,mbID,artistName)))
        searchedForErrors[artistID] = True
        apiio.sleep(15)
        nErr.append(artistID)
        if len(nErr) >= 6:
            break
        continue

    nErr = []
    searchedForMasterArtistsData[artistID] = response
    searchedForMasterArtists[artistID] = artistName
    apiio.sleep(20)
    n += 1
        
    if n % 5 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        errors.save(data=searchedForErrors)
        print("="*150)
        apiio.wait(10)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
            
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)
if len(nErr) > 0:
    errors.save(data=searchedForErrors)
    for artistID in nErr:
        print("del searchedForErrors['{0}']".format(artistID))
    print("errors.save(data=searchedForErrors)")

In [ ]:

del searchedForErrors['124024766635548005270309261423453479764']
del searchedForErrors['237748617780344156692083260097110805341']
del searchedForErrors['231184520235867051473735912378686360891']
del searchedForErrors['2875464283021047873405831377507163865']
del searchedForErrors['219261416572199174528917937125268559566']
del searchedForErrors['191092118056524447748174293367409296013']
errors.save(data=searchedForErrors)

## Save Results

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = Series(mad).apply(Series)
        #df = DataFrame({v['mbid']: {k2: v2 for k2,v2 in v.items() if k2 not in []} for k,v in mad.items()}).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)

if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = setlistfm.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF.drop_duplicates(keep='first')
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("Saving Data")
    setlistfm.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [ ]:
mio   = setlistfm.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [ ]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

## Download The Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "12:05pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(6.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

# Website Download

In [6]:
from apiutils import WebIO
from ioutils import HTMLIO, FileIO
from bs4 import element
wio = WebIO()
hio = HTMLIO()
io  = FileIO()

In [7]:
webArtists = MusicDBData(path=permDir, fname="{0}SearchedForWebArtists".format(db.lower()))
webArtistsData = MusicDBData(path=permDir, fname="{0}SearchedForWebArtistsData".format(db.lower()))

## Starter

In [ ]:
modValRefs  = {}
modValLinks = []
for modVal in modVals:
    url = "https://www.setlist.fm/artist/browse/{0}/1.html".format(modVal)
    savename = "{0}.p".format(modVal)
    retval = wio.get(url)
    print(url)
    io.save(idata=retval.getData(), ifile=savename)
    wio.sleep(5)

In [ ]:
modValListRefs = {}
for modVal in modVals:
    savename = "{0}.p".format(modVal)
    bsdata = hio.get(io.get(savename))
    
    submodListDiv  = bsdata.find("div", {"id": "ide"})
    submodListRefs = [li.find('a') for li in submodListDiv.findAll("li")] if isinstance(submodListDiv, element.Tag) else []
    submodListRefs = {ref.get('href'): ref.text for ref in submodListRefs if (isinstance(ref, element.Tag) and "/setlists/" in ref.get('href'))}
    modValListRefs.update(submodListRefs)

## Download ModVal Refs

In [58]:
from string import ascii_lowercase
modVals = [ch for ch in ascii_lowercase] + ['0-9']

modValRefs = {}
for modVal in modVals:
    savename = "{0}.p".format(modVal)
    bsdata = hio.get(io.get(savename))
    modListUL   = bsdata.find("ul", {"class": "row"})
    modListRefs = [li.find('a') for li in modListUL.findAll("li")] if isinstance(modListUL, element.Tag) else []
    modValRefs.update({ref.get('href'): ref.text for ref in modListRefs if isinstance(ref, element.Tag)})
modValRefs = Series(modValRefs)

searchedForWebArtists = webArtists.get()
modValRefsToGet = modValRefs[~modValRefs.index.isin(searchedForWebArtists.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   All ModVal Refs:    {0}".format(modValRefs.shape[0]))
print("   Known ModVal Refs:  {0}".format(len(searchedForWebArtists)))
print("   ModVal Refs To Get: {0}".format(len(modValRefsToGet)))

SetListFM Search Results
   All ModVal Refs:    4522
   Known ModVal Refs:  4522
   ModVal Refs To Get: 0


In [57]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
localWebArtistsDict  = webArtists.get()
modValListRefs = {}

for i,(ref,name) in enumerate(modValRefsToGet.iteritems()):
    if localWebArtistsDict.get(ref) is not None:
        continue
        
    url = ref.replace("../../../", "https://www.setlist.fm/")
    try:
        print("Getting {0: <50}".format(url), end="\t")
        response = wio.get(url)
    except:
        print("ERROR Downloading! Stopping.")
        break
        

    try:
        bsdata = hio.get(response.getData())
    except:
        print("ERROR Creating BeautifulSoup! Stopping.")
        break
        
    try:
        submodListDiv  = bsdata.find("div", {"id": "ide"})
        submodListRefs = [li.find('a') for li in submodListDiv.findAll("li")] if isinstance(submodListDiv, element.Tag) else []
        submodListRefs = {ref.get('href'): ref.text for ref in submodListRefs if (isinstance(ref, element.Tag) and "/setlists/" in ref.get('href'))}
    except:
        print("ERROR Getting Refs From BeautifulSoup! Stopping.")
        break
        
    print(len(submodListRefs))
    modValListRefs.update(submodListRefs)    
    localWebArtistsDict[ref] = True
    wio.sleep(5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localWebArtistsDict), db))
        webArtists.save(data=localWebArtistsDict)
        
        previousData = webArtistsData.get()
        print("Found {0} Previous Web Links".format(len(previousData)))
        print("Found {0} New Web Links".format(len(modValListRefs)))
        newData = {**previousData, **modValListRefs}
        print("Found {0} Total Web Links".format(len(newData)))
        webArtistsData.save(data=newData)
        modValListRefs = {}
        
        print("="*150)
        wio.wait(10)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localWebArtistsDict), db))
webArtists.save(data=localWebArtistsDict)

previousData = webArtistsData.get()
print("Found {0} Previous Web Links".format(len(previousData)))
print("Found {0} New Web Links".format(len(modValListRefs)))
newData = {**previousData, **modValListRefs}
print("Found {0} Total Web Links".format(len(newData)))
webArtistsData.save(data=newData)


Process [Getting SetListFM Artist Data] Start    ==> Time Is 2022-05-03 07:51:36
========================= termTime(day=today,time=9:00pm) =========================
   ====> Terminate Time Set To 2022-05-03 21:00:00 <====
   ====> Will Terminate Process 13 Hours and 8 Minutes From Now
Getting https://www.setlist.fm/artist/browse/l/176.html   	60
Getting https://www.setlist.fm/artist/browse/m/299.html   	60
Getting https://www.setlist.fm/artist/browse/c/236.html   	60
Getting https://www.setlist.fm/artist/browse/m/55.html    	60
Getting https://www.setlist.fm/artist/browse/s/229.html   	60
Getting https://www.setlist.fm/artist/browse/p/192.html   	60
Getting https://www.setlist.fm/artist/browse/m/267.html   	60
Getting https://www.setlist.fm/artist/browse/s/297.html   	60
Getting https://www.setlist.fm/artist/browse/p/8.html     	60
Getting https://www.setlist.fm/artist/browse/s/417.html   	60
Getting https://www.setlist.fm/artist/browse/t/178.html   	60
Getting https://www.setlist.fm/a

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting https://www.setlist.fm/artist/browse/v/43.html    	60
Getting https://www.setlist.fm/artist/browse/f/76.html    	60
Getting https://www.setlist.fm/artist/browse/t/39.html    	60
Getting https://www.setlist.fm/artist/browse/z/12.html    	60
Getting https://www.setlist.fm/artist/browse/k/32.html    	60
Getting https://www.setlist.fm/artist/browse/d/185.html   	60
Getting https://www.setlist.fm/artist/browse/v/88.html    	60
Getting https://www.setlist.fm/artist/browse/w/69.html    	60
Getting https://www.setlist.fm/artist/browse/r/94.html    	60
Getting https://www.setlist.fm/artist/browse/p/115.html   	60
Getting https://www.setlist.fm/artist/browse/l/55.html    	60
Getting https://www.setlist.fm/artist/browse/h/159.html   	60
Getting https://www.setlist.fm/artist/browse/t/44.html    	60
Getting https://www.setlist.fm/artist/browse/m/317.html   	60
Getting https://www.setlist.fm/artist/browse/i/12.html    	60
Getting https://www.setlist.fm/artist/browse/s/412.html   	60
Getting 

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting https://www.setlist.fm/artist/browse/b/230.html   	60
Getting https://www.setlist.fm/artist/browse/0-9/26.html  	60
Getting https://www.setlist.fm/artist/browse/l/68.html    	60
Getting https://www.setlist.fm/artist/browse/k/150.html   	60
Getting https://www.setlist.fm/artist/browse/s/234.html   	60
Getting https://www.setlist.fm/artist/browse/m/52.html    	60
Getting https://www.setlist.fm/artist/browse/a/248.html   	51
Getting https://www.setlist.fm/artist/browse/s/94.html    	60
Getting https://www.setlist.fm/artist/browse/i/46.html    	60
Getting https://www.setlist.fm/artist/browse/o/10.html    	60
Getting https://www.setlist.fm/artist/browse/t/147.html   	60
Getting https://www.setlist.fm/artist/browse/m/341.html   	60
Getting https://www.setlist.fm/artist/browse/r/18.html    	60
Getting https://www.setlist.fm/artist/browse/v/53.html    	60
Getting https://www.setlist.fm/artist/browse/h/155.html   	60
Getting https://www.setlist.fm/artist/browse/p/155.html   	60
Getting 

...

Getting https://www.setlist.fm/artist/browse/j/81.html    	60
Getting https://www.setlist.fm/artist/browse/d/261.html   	60
Getting https://www.setlist.fm/artist/browse/g/45.html    	60
Getting https://www.setlist.fm/artist/browse/m/246.html   	60
Getting https://www.setlist.fm/artist/browse/s/2.html     	60
Getting https://www.setlist.fm/artist/browse/u/12.html    	60
Getting https://www.setlist.fm/artist/browse/t/130.html   	60
Getting https://www.setlist.fm/artist/browse/l/119.html   	60
Getting https://www.setlist.fm/artist/browse/c/153.html   	60
Getting https://www.setlist.fm/artist/browse/d/174.html   	60
Getting https://www.setlist.fm/artist/browse/s/449.html   	60
Getting https://www.setlist.fm/artist/browse/a/58.html    	60
Getting https://www.setlist.fm/artist/browse/t/232.html   	60
Getting https://www.setlist.fm/artist/browse/d/125.html   	60
Getting https://www.setlist.fm/artist/browse/m/99.html    	60
Getting https://www.setlist.fm/artist/browse/v/25.html    	60
Getting 

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting https://www.setlist.fm/artist/browse/m/352.html   	60
Getting https://www.setlist.fm/artist/browse/m/204.html   	60
Getting https://www.setlist.fm/artist/browse/m/62.html    	60
Getting https://www.setlist.fm/artist/browse/d/277.html   	60
Getting https://www.setlist.fm/artist/browse/a/118.html   	60
Getting https://www.setlist.fm/artist/browse/l/141.html   	60
Getting https://www.setlist.fm/artist/browse/a/116.html   	60
Getting https://www.setlist.fm/artist/browse/c/109.html   	60
Getting https://www.setlist.fm/artist/browse/s/185.html   	60
Getting https://www.setlist.fm/artist/browse/b/21.html    	60
Getting https://www.setlist.fm/artist/browse/t/105.html   	60
Getting https://www.setlist.fm/artist/browse/s/302.html   	60
Getting https://www.setlist.fm/artist/browse/s/392.html   	60
Getting https://www.setlist.fm/artist/browse/t/35.html    	60
Getting https://www.setlist.fm/artist/browse/r/196.html   	60
Getting https://www.setlist.fm/artist/browse/c/208.html   	60
Getting 

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting https://www.setlist.fm/artist/browse/t/132.html   	60
Getting https://www.setlist.fm/artist/browse/s/164.html   	60
Getting https://www.setlist.fm/artist/browse/n/92.html    	60
Getting https://www.setlist.fm/artist/browse/j/101.html   	60
Getting https://www.setlist.fm/artist/browse/y/17.html    	60
Getting https://www.setlist.fm/artist/browse/b/146.html   	60
Getting https://www.setlist.fm/artist/browse/b/105.html   	60
Getting https://www.setlist.fm/artist/browse/c/143.html   	60
Process [Getting SetListFM Artist Data] Ran For 36 Minutes and 56 Seconds    ==> Time Is 2022-05-03 08:28:33
Saving 4522 SetListFM Searched For Artist (Info) IDs
Found 269251 Previous Web Links
Found 480 New Web Links
Found 269714 Total Web Links


In [48]:
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localWebArtistsDict), db))
webArtists.save(data=localWebArtistsDict)

previousData = webArtistsData.get()
print("Found {0} Previous Web Links".format(len(previousData)))
print("Found {0} New Web Links".format(len(modValListRefs)))
newData = {**previousData, **modValListRefs}
print("Found {0} Total Web Links".format(len(newData)))
webArtistsData.save(data=newData)


Saving 4164 SetListFM Searched For Artist (Info) IDs
Found 247792 Previous Web Links
Found 1260 New Web Links
Found 249035 Total Web Links


# Download Artist Refs

In [107]:
from pandas import merge
pdbio = PanDBIO()
mmeDF = pdbio.getData()
panData = mmeDF["ArtistName"].reset_index().copy(deep=True)
tmp   = mmeDF[mmeDF["SetListFM"].notna()][["ArtistName", "SetListFM"]]
known = Series(tmp["ArtistName"].values, index=tmp["SetListFM"].values)

In [98]:
webArtistsDataDict = webArtistsData.get()
setlistWebRefs = DataFrame(Series(webArtistsDataDict, name="ArtistName"))
setlistWebRefs.index.name = "SetListFM"
setlistWebRefs = setlistWebRefs.reset_index()
setlistWebRefs["ID"] = setlistWebRefs["SetListFM"].apply(lambda x: x.split("-")[-1][:-5])
def fixName(x):
    vals = x.split(", ")
    if len(vals) == 2:
        return " ".join([vals[1], vals[0]])
    return x
setlistWebRefs["ArtistName"] = setlistWebRefs["ArtistName"].apply(fixName)
knownSetListFMArtists = merge(panData, setlistWebRefs, on="ArtistName", how='left')


In [149]:
allSetListFM     = knownSetListFMArtists[knownSetListFMArtists["SetListFM"].notna()]
localArtistsDict = localArtists.get()
artistNamesToGet = allSetListFM[~allSetListFM["ID"].isin(localArtistsDict.keys())]

print("{0} Search Results".format(db))
print("   All Artist Refs:    {0}".format(allSetListFM.shape[0]))
print("   Known Artist Refs:  {0}".format(len(localArtistsDict)))
print("   Artist Refs To Get: {0}".format(len(artistNamesToGet)))

SetListFM Search Results
   All Artist Refs:    280328
   Known Artist Refs:  29035
   Artist Refs To Get: 247059


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
localArtistsDataDict = localArtistsData.get()
searchedForErrors    = errors.get()
nErr = []

for i,(idx,row) in enumerate(artistNamesToGet.iterrows()):        
    artistID   = row["ID"]
    artistName = row["ArtistName"]
    artistURL  = row["SetListFM"].replace("../../../", "https://www.setlist.fm/")
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    print("Getting {0: <50}".format("{0} ({1})".format(artistName,artistID)), end="\t")
    
    try:
        response = wio.get(artistURL)
    except:
        print("ERROR Downloading! Stopping.")
        wio.sleep(15)
        nErr.append(artistID)
        searchedForErrors[artistID] = True
        if len(nErr) >= 6:
            break
        continue

    try:
        bsdata = hio.get(response.getData())
    except:
        print("ERROR Creating BeautifulSoup! Stopping.")
        wio.sleep(15)
        nErr.append(artistID)
        searchedForErrors[artistID] = True
        if len(nErr) >= 6:
            break
        continue
        
    try:
        artistLinks  = bsdata.find("div", {"class": "artistLinks"})
        externalRefs = {ref.get('href'): ref.text.strip() for ref in artistLinks.findAll("a")} if isinstance(artistLinks, element.Tag) else {}
        mbDiv = bsdata.find("div", {"class": "info"})
        mbid  = mbDiv.find('span').text if isinstance(mbDiv, element.Tag) else None
        artistData = {"MBID": mbid, "Refs": externalRefs}
    except:
        print("ERROR Getting Refs From BeautifulSoup! Stopping.")
        wio.sleep(15)
        nErr.append(artistID)
        searchedForErrors[artistID] = True
        if len(nErr) >= 6:
            break
        continue
        
    
    print("{0}  ({1})".format(artistData["MBID"], len(artistData["Refs"])))
    localArtistsDict[artistID] = artistName
    localArtistsDataDict[artistID] = artistData
    nErr = []
    wio.sleep(5)
    n += 1
    
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        localArtistsData.save(data=localArtistsDataDict)
        errors.save(data=searchedForErrors)
        print("="*150)
        wio.wait(10)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

if True:
    ts.stop()
    print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
    localArtists.save(data=localArtistsDict)
    localArtistsData.save(data=localArtistsDataDict)
    errors.save(data=searchedForErrors)
    if len(nErr) > 0:
        for artistID in nErr:
            print("del searchedForErrors['{0}']".format(artistID))
        print("errors.save(data=searchedForErrors)")

Process [Getting SetListFM Artist Data] Start    ==> Time Is 2022-05-03 12:10:11
========================= termTime(day=today,time=7:00pm) =========================
   ====> Terminate Time Set To 2022-05-03 19:00:00 <====
   ====> Will Terminate Process 6 Hours and 49 Minutes From Now
Getting Placebo (3bd89c48)                                	979a39b4-2df9-4767-a3ca-7940e8aff05a  (1)
Getting Barbara (4bda0fc6)                                	286c9551-d703-4037-a022-b1b146afbabb  (1)
Getting Gentleman (63d11eaf)                              	e25193d9-a658-43eb-9fa3-36c16435c403  (1)
Getting The Shadows (23dda46b)                            	106eb6ac-d889-4189-a2fa-6e01604df6f2  (1)
Getting The Shadows (23d4c067)                            	16670dd6-ca14-4881-a066-66e45ac22399  (1)
Getting Metric (43f3e377)                                 	1d72bb5d-0dc0-4a2e-a7e7-fd59d6341ade  (1)
Getting Roy Harper (23f4c05b)                             	c7ea266e-d9c4-4d88-9044-45cd2aba6541  (1)
Getting

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting White Lies (43f4cf83)                             	c251c2ee-1300-4ed2-bfb1-91b5f181a7af  (1)
Getting White Lies (4bc14f7a)                             	5cd124e2-b3cb-47b5-8b9d-644f0f8c141b  (1)
Getting Lena (53d38ba9)                                   	97bd413d-1818-454d-8ec1-c810c042ffc0  (1)
Getting Lena (3bd04004)                                   	1a82c7d5-4b47-4a27-8ed5-59f7346386f0  (1)
Getting Jimmy Barnes (2bc7ccbe)                           	8d147e1f-19a0-42fa-bfce-2ea5975aaf35  (1)
Getting Riff Raff (5bdd17c4)                              	08224fb1-5933-4997-a16b-d6d57ddad9d7  (1)
Getting Riff Raff (43d657c3)                              	df64cd25-0d23-4f4c-b6b0-6a49bc69a6df  (1)
Getting Riff Raff (3c45933)                               	1447c361-0b1c-40d6-8bfd-9252d4451ad8  (1)
Getting Riff Raff (3bded434)                              	78ca4da6-3a89-42d0-aff3-369830435337  (1)
Getting Riff Raff (4bd007f6)                              	ae6d2b1e-3632-4ec0-a86d-24d87d45

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting Hole (bc7a10e)                                    	f60b8297-a982-405b-b09b-bab2f58c50ed  (1)
Getting Kalash (bcacd5a)                                  	958a5d25-43b9-4258-8249-e8a2e3a1b833  (1)
Getting Velvet Underground (63c502c7)                     	54aa91b4-7099-4461-8873-e6768226e61f  (1)
Getting The Band (1bcbe5a0)                               	712463f7-4e96-44b0-9a0b-efe42f208820  (1)
Getting Tiamat (73c94e29)                                 	401cf336-c0c6-4350-b821-ba9732973f64  (1)
Getting Overkill (53cdb769)                               	7de71e63-b419-4ab0-acba-8db2ecda2764  (1)
Getting Nilsson (4bd41f02)                                	3c6a438b-463e-4a40-a5cd-a15149478eab  (2)
Getting UFO (3bd6acfc)                                    	c827b8a7-0372-4711-9777-532b387e834b  (1)
Getting UFO (2bd6acf2)                                    	f1c2ba3c-2c96-4988-9b2c-5da86c14c383  (1)
Getting Jay Rock (5bd65344)                               	fc5145f0-558d-452a-8d6e-82eeab1b

...

Getting Ghost (3d46923)                                   	97ea6a8e-2011-45f0-895e-8bb082b1aa64  (2)
Getting Exile (63d4a24f)                                  	9580ea3d-a132-43f7-878a-b46e2be4e822  (2)
Getting Exile (1bd41d20)                                  	bceba9d9-3a3c-4883-af65-e44bd5848169  (2)
Getting Exile (3d41d27)                                   	730d0557-1abb-4310-8d99-66b61b43c3b0  (1)
Getting Exile (bf7d956)                                   	cafa1ff4-fa45-4f8b-b0fe-9cb759b24925  (1)
Getting Chris Andrews (3d65583)                           	6606ea78-d757-4070-95d5-feac46c561b4  (1)
Getting Wigwam (53c8737d)                                 	22c3c917-ae3f-4bc5-a609-d34766ac55a2  (1)
Getting The Wake (63d4da9f)                               	b52bd834-ebdb-4b9a-9bff-1d36b654a084  (2)
Getting The Wake (63de161b)                               	a806b588-20bd-479e-8ea4-0fe07cf3b13c  (2)
Getting The Wake (4bfedf22)                               	6f251694-1d8e-439b-9c8f-b56e22b6

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting TNT (43d4d3d7)                                    	c557f8ec-0279-48bf-ba97-a3758129f195  (1)
Getting TNT (43d68bf3)                                    	3c2e5961-a954-4b95-94c7-3b48415165cb  (1)
Getting TNT (4bd68bf2)                                    	d8d9f258-a27d-4c59-adfc-427f9f3944b3  (1)
Getting TNT (53d68bf1)                                    	71ef4ef6-b35c-4876-afba-693d568b3938  (1)
Getting TNT (43d68bf7)                                    	e0255228-30ff-4e2f-be50-23419f22dd7a  (1)
Getting TNT (5bd68bf4)                                    	458f0043-e94f-43e3-bb73-38bcb1ce7e0a  (1)
Getting TNT (4bd2770a)                                    	3e8d3f6c-94ad-4ddb-bf0a-124d834b1487  (1)
Getting Jorge Reyes (1bd191a0)                            	ea854440-830d-4767-a038-7d9ae6f386e7  (1)
Getting Ceremony (13d009a5)                               	26486398-3b40-4f35-902c-744acc94078b  (1)
Getting Ceremony (13d00989)                               	21cf55f9-37be-4586-9117-2b5af867

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting Danger (23df24b3)                                 	30be9fa6-ef7e-4987-8cd6-e461a701be50  (1)
Getting Danger (2bd628e2)                                 	6f65463d-d790-4899-814e-f6ff8601006d  (1)
Getting Ego (5bceefb8)                                    	eaa29509-0b88-42c9-971e-cabef679cf76  (1)
Getting Ego (5bd7e7d8)                                    	87a06a2e-9060-4055-af14-38a28880b1b7  (1)
Getting Ego (4bd7e7de)                                    	44e3eec6-fba7-4578-98e3-91b1ff9321c6  (1)
Getting Aube (23d6c867)                                   	c57d8c1f-e8a9-442e-9700-1821f165460a  (1)
Getting Aube (4bddcf0e)                                   	dc775fa8-a94c-475c-a0b0-fddc93c75a7b  (2)
Getting Francis Lalanne (3bd4d0f4)                        	6de79338-11ee-4d3a-ada2-b065f30c7495  (2)
Getting Francis Lalanne (3bda5418)                        	54cb0e89-17b9-4dac-a54d-55f9483911d0  (1)
Getting Owen Gray (4bd63bbe)                              	ee11baba-0b24-428b-9fec-396662f4